In [1]:
# Copyright 2024 - Andrew Kwok Fai LUI, 
# Robotics and Autonomous Systems Group, REF, RI
# and the Queensland University of Technology

__author__ = 'Andrew Lui'
__copyright__ = 'Copyright 2024'
__license__ = 'GPL'
__version__ = '1.0'
__email__ = 'ak.lui@qut.edu.au'
__status__ = 'Development'

import os, math, yaml, numbers, pickle
from enum import Enum
from collections import defaultdict
from datetime import datetime
import cv2
import numpy as np

import ipywidgets as widgets
from IPython.display import display, clear_output

from cgras.detector.models.detect import CoralObject, test_load_coral_object_detect_model
from cgras.detector.models import logger

In [2]:
def test_build_spatial_distribute_model():
    logdata_folder = '/home/qcr/cgras_data/detector/combined_new/'
    params = {
        'logdata_folder': logdata_folder, 
        'reco_model_filename': 'reco_model.yaml',
        'loctile_model_filename': 'loctile_model.yaml',
        'yolo_model_file': '/home/qcr/cgras_data/YoloModel/20240923_tiledimages_yolov8xseg_naive.pt',
        'cod_model_filename': 'coral_object_detect_model.yaml', 
        'cod_debug_blob_images': True,
        'cod_blob_size': (640, 640),
        'cod_blob_overlap_pix': 32,
        'cod_use_cached_object_detection': False,
        'cod_duplicate_max_displacement_images': 16,
        'cod_duplicate_max_displacement_blobs': 32,        
    } 
    logger.info('Loading CoralObjectDetect Model from a yaml file')
    cod_model = test_load_coral_object_detect_model(params)
    cod_model.print_info()
    return cod_model

logdata_folder = logdata_folder = '/home/qcr/cgras_data/detector/combined_new/'
cod_model = test_build_spatial_distribute_model()



[INFO] [1727256643.352144]: Loading CoralObjectDetect Model from a yaml file
[INFO] [1727256645.786279]: Number of objects: 2001
[INFO] [1727256645.787352]: Number of invalidated objects: 116
[INFO] [1727256645.787975]: Number of unique objects: 1885
[INFO] [1727256645.788260]: Tile size: (21740, 8764)
[INFO] [1727256645.788682]: Object Class Names: {0: 'recruit_live_white', 1: 'recruit_cluster_live_white', 2: 'recruit_symbiotic', 3: 'recruit_cluster_symbiotic', 4: 'recruit_partial', 5: 'recruit_cluster_partial', 6: 'recruit_dead', 7: 'recruit_cluster_dead', 8: 'grazer_snail', 9: 'pest_tubeworm', 10: 'unknown'}


In [3]:
import plotly.express as px

class HeatmapModel():
    def __init__(self, object_list:list, **kwargs):
        # input parameters
        self.object_list = object_list
        # other keyword parameters - output cached data and debug information
        self.logdata_folder = kwargs.get('logdata_folder', None)
        self.use_heatmap_cache_file = kwargs.get('use_heatmap_cache_file', True)        
        # model variables
        self.heatmap_cache = dict()
    
    def generate_heatmap(self, map_size:tuple, class_filter:tuple=None, include_invalidated=False):
        cache_index = (class_filter, map_size[0], map_size[1])
        if cache_index in self.heatmap_cache:
            print('using cache')
            return self.heatmap_cache[cache_index]
        if class_filter is not None and type(class_filter) == str:
            class_filter = [class_filter]
        map_array = np.zeros(shape=(map_size[1], map_size[0]), dtype=np.uint16)
        coral_object:CoralObject
        for coral_object in self.object_list:
            if coral_object.invalidated and not include_invalidated:
                continue
            if class_filter is not None and coral_object.cls_name not in class_filter:
                continue
            x, y = int(coral_object.centre_normalized[0] * map_size[0]), int(coral_object.centre_normalized[1] * map_size[1])
            x, y = min(x, map_size[0] - 1), min(y, map_size[1] - 1)
            map_array[y, x] += 1
        self.heatmap_cache[cache_index] = map_array
        return map_array
    
class HeatmapHelper():
    """ HeatmapHelper provides functions to help create heatmaps from a list of CoralObjects

    """
    @staticmethod
    def compute_object_count_map(object_list:list, map_size:tuple, class_filter=None, include_invalidated=False) -> np.ndarray:
        """ compute a 2d object count map as a numpy array from the locatios of the CoralObject in the input object_list

        :param object_list: the input list of CoralObjects
        :type object_list: list
        :param map_size: the dimension (xdim, ydim) of the map 
        :type map_size: tuple
        :param class_filter: a list of classes or a single class to be included in the count, defaults to None
        :type class_filter: Any, optional
        :param include_invalidated: include invalidated objects in the counting, defaults to False
        :type include_invalidated: bool, optional
        :return: the count map as a 2d numpy array
        :rtype: np.ndarray
        """
        if class_filter is not None and type(class_filter) == str:
            class_filter = [class_filter]
        count_map_array = np.zeros(shape=(map_size[1], map_size[0]), dtype=np.uint16)
        coral_object:CoralObject
        for coral_object in object_list:
            if coral_object.invalidated and not include_invalidated:
                continue
            if class_filter is not None and coral_object.cls_name not in class_filter:
                continue
            x, y = int(coral_object.centre_normalized[0] * map_size[0]), int(coral_object.centre_normalized[1] * map_size[1])
            x, y = min(x, map_size[0] - 1), min(y, map_size[1] - 1)
            count_map_array[y, x] += 1
        return count_map_array  
    
    @staticmethod
    def generate_plotly_heatmap(count_map_array:np.ndarray, fig_size:tuple, show_fig:bool=False, output_file:str=None):
        """ returns a plotly figure object containing the heatmap generated from the given object count map

        :param count_map_array: the coral object count map to be converted into a graphical heatmap
        :type count_map_array: np.ndarray
        :param fig_size: the size (xdin, ydim) of the graphical heatmap in pixels
        :type fig_size: tuple
        :param show_fig: to call fig.show() at the end of the function, defaults to False
        :type show_fig: bool, optional
        :param output_file: the path where the figure is saved to an image file if provided, defaults to None
        :type output_file: str, optional
        :return: a plotly fig object
        :rtype: plotly.graph_objs._figure.Figure
        """
        fig = px.imshow(count_map_array, text_auto=True)
        fig.update_layout(width=fig_size[0], height=fig_size[1])
        if output_file is not None:
            fig.write_image(output_file)
        if show_fig:
            fig.show()
        return fig

In [4]:
object_list = cod_model.get_object_list()


In [7]:


class_names = list(cod_model.get_object_class_names().values())
class_names.insert(0, 'all')
w = widgets.Dropdown(
    options=class_names,
    description='Object Class: ',
)

def plot_heatmap(z, fig_size=(2000, 1000)):
    fig = px.imshow(z, text_auto=True)
    fig.update_layout(width=fig_size[0], height=fig_size[1])
    fig.show()
    
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_class = change['new']
        if selected_class == 'all':
            selected_class = None
        count_map_array = HeatmapHelper.compute_object_count_map(object_list, map_size, selected_class)
        clear_output(wait=True)
        display(w)
        HeatmapHelper.generate_plotly_heatmap(count_map_array, fig_size, show_fig=True)
        
map_size = (30, 10)
fig_size = (1500, 500)
w.observe(on_change)
display(w)


Dropdown(description='Object Class: ', index=1, options=('all', 'recruit_live_white', 'recruit_cluster_live_wh…